# Validasi Silang

Estimasi waktu: **45 menit**

Dalam analisis supervised learning, performa model harus diukur pada data yang tidak dipakai saat pelatihan. Bila hanya ada satu pembagian data, hasil penilaian bisa sangat bergantung pada baris mana yang kebetulan masuk data uji. Validasi silang membagi data menjadi beberapa bagian dan menggilir peran masing-masing, sehingga penilaian lebih stabil.

## Tujuan

- membagi data menjadi bagian latih, validasi, dan uji;
- menilai model dengan `cross_val_score` dan `KFold`;
- memilih fitur tunggal terbaik;
- merangkai langkah pengolahan dengan `Pipeline`.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

warnings.filterwarnings("ignore")

## Membaca data

Data yang dipakai adalah data penjualan mobil yang sudah di-encode menjadi angka semua. Berkas diunduh sekali, lalu dibaca dengan pandas.

In [ ]:
data_path = Path("data/encoded_car_data.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/encoded_car_data.csv"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

cars = pd.read_csv(data_path)
cars.head()

In [ ]:
cars.dtypes.value_counts()

In [ ]:
cars.info()

Semua kolom sudah bertipe numerik, jadi tidak ada encoding tambahan yang diperlukan.

## Menyiapkan data

Fitur dan target dipisahkan, lalu data dibagi 70% latih dan 30% uji.

In [ ]:
features = cars.drop(columns=["price"])
target = cars["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=42
)

## Membangun dan menilai model

Model regresi linear dilatih pada data latih, lalu dinilai pada data uji.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("R2 data latih:", model.score(X_train, y_train))
print("R2 data uji  :", model.score(X_test, y_test))

Skor data latih sekitar 0,93, artinya 93% variasi harga bisa dijelaskan model pada data yang dipakai saat pelatihan. Skor data uji turun ke sekitar 0,85. Selisih ini wajar, dan angka data uji yang dipakai sebagai ukuran performa sebenarnya.

In [ ]:
print("R2  :", r2_score(y_true=y_test, y_pred=predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_true=y_test, y_pred=predictions)))

RMSE memberi galat dalam satuan harga, sehingga lebih mudah dibaca daripada MSE.

### Contoh prediksi

Beberapa baris pertama data dipakai untuk membandingkan nilai prediksi dengan nilai aktual.

In [ ]:
sample_features = features.iloc[:3]
sample_labels = target.iloc[:3]

print("Prediksi:", model.predict(sample_features))
print("Aktual  :", list(sample_labels))

### Pipeline

`Pipeline` menjalankan standardisasi lalu regresi secara berurutan dalam satu objek.

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("regression", LinearRegression())])
pipe.fit(X_train, y_train)

print("R2 data latih:", pipe.score(X_train, y_train))
print("R2 data uji  :", pipe.score(X_test, y_test))

## Latihan 1

Buat pipeline bernama `pipe_normalizer` yang memakai `Normalizer` sebagai pengganti `StandardScaler`, lalu hitung R kuadrat pada data latih dan uji.

In [ ]:
pipe_normalizer = Pipeline([("normalizer", Normalizer()), ("regression", LinearRegression())])
pipe_normalizer.fit(X_train, y_train)

print("R2 data latih:", pipe_normalizer.score(X_train, y_train))
print("R2 data uji  :", pipe_normalizer.score(X_test, y_test))

`Normalizer` menyetel setiap baris agar berpanjang satu, berbeda dari `StandardScaler` yang menyetel setiap kolom. Karena representasi datanya berubah, skornya pun berbeda: pada pembagian data ini skor uji Normalizer justru lebih tinggi, sekitar 0,88.

## Satu fitur

Model juga bisa dilatih memakai satu fitur saja, lalu fitur dengan skor tertinggi dipilih.

In [ ]:
feature_names = list(features)
feature_names

In [ ]:
single_feature_scores = []
pipe = Pipeline([("scaler", StandardScaler()), ("regression", LinearRegression())])

for feature in feature_names:
    pipe.fit(X_train[[feature]], y_train)
    single_feature_scores.append(pipe.score(X_train[[feature]], y_train))

In [ ]:
plt.bar(feature_names, single_feature_scores)
plt.xticks(rotation=90)
plt.ylabel("R squared")
plt.tight_layout()
plt.show()

In [ ]:
best_feature = feature_names[int(np.argmax(single_feature_scores))]
print("Fitur terbaik:", best_feature)

pipe.fit(features[[best_feature]], target)
print("R2 pada seluruh data:", pipe.score(features[[best_feature]], target))

Fitur `enginesize` memberi R kuadrat tertinggi, sehingga ukuran mesin paling kuat menjelaskan harga mobil.

## Latihan 2

Cari fitur terbaik memakai data uji, tanpa standardisasi.

In [ ]:
test_scores = []

for feature in feature_names:
    plain_model = LinearRegression()
    plain_model.fit(X_train[[feature]], y_train)
    test_scores.append(plain_model.score(X_test[[feature]], y_test))

best_plain_feature = feature_names[int(np.argmax(test_scores))]
print("Fitur terbaik pada data uji:", best_plain_feature)
print("R2 tertinggi:", max(test_scores))

Memilih fitur berdasarkan data uji membuat skor uji ikut terpakai saat pemilihan, sehingga angkanya cenderung terlalu optimistis. Praktik yang benar adalah memilih fitur memakai data latih atau validasi.

## Validasi silang K-fold

`cross_val_score` membagi data menjadi beberapa lipatan dan mengembalikan satu skor per lipatan.

In [ ]:
print("Jumlah baris:", len(features))

In [ ]:
plain_model = LinearRegression()
fold_scores = cross_val_score(plain_model, features, target, scoring="r2", cv=3)
fold_scores

In [ ]:
def describe_scores(scores):
    print("Skor       :", scores)
    print("Rata-rata  :", scores.mean())
    print("Simpangan  :", scores.std())


describe_scores(fold_scores)

Sebaran skor antar-lipatan cukup lebar, bahkan ada lipatan yang skornya negatif. Artinya model gagal pada sebagian partisi data, sehingga satu angka rata-rata saja bisa menyesatkan. Simpangan baku skor perlu ikut dilaporkan.

## Latihan 3

Hitung skor validasi silang dengan 5 lipatan memakai model yang sama.

In [ ]:
five_fold_scores = cross_val_score(plain_model, features, target, scoring="r2", cv=5)
describe_scores(five_fold_scores)

## KFold manual

`KFold` memberi kendali penuh atas indeks latih dan uji setiap lipatan, misalnya ketika model tidak tersedia langsung di scikit-learn.

In [ ]:
n_splits = 2
kf = KFold(n_splits=n_splits)
fold_r2 = np.zeros(n_splits)

pipe = Pipeline([("scaler", StandardScaler()), ("regression", LinearRegression())])

for fold_index, (train_index, test_index) in enumerate(kf.split(features, target)):
    print("LATIH:", train_index)
    print("UJI  :", test_index)

    X_tr, X_te = features.iloc[train_index], features.iloc[test_index]
    y_tr, y_te = target[train_index], target[test_index]

    pipe.fit(X_tr, y_tr)
    fold_r2[fold_index] = pipe.score(X_te, y_te)

fold_r2.mean()

In [ ]:
n_splits = 3
kf = KFold(n_splits=n_splits)
fold_r2 = np.zeros(n_splits)

for fold_index, (train_index, test_index) in enumerate(kf.split(features, target)):
    X_tr, X_te = features.iloc[train_index], features.iloc[test_index]
    y_tr, y_te = target[train_index], target[test_index]

    pipe.fit(X_tr, y_tr)
    fold_r2[fold_index] = pipe.score(X_te, y_te)

fold_r2.mean()

Dengan tiga lipatan, dua pertiga data dipakai untuk latihan pada setiap putaran.

## Latihan 4

Buat `KFold` dengan tiga lipatan dan pengacakan aktif, lalu hitung rata-rata R kuadratnya.

In [ ]:
shuffled_kf = KFold(n_splits=3, shuffle=True, random_state=42)
shuffled_scores = np.zeros(3)

for fold_index, (train_index, test_index) in enumerate(shuffled_kf.split(features, target)):
    X_tr, X_te = features.iloc[train_index], features.iloc[test_index]
    y_tr, y_te = target[train_index], target[test_index]

    pipe.fit(X_tr, y_tr)
    shuffled_scores[fold_index] = pipe.score(X_te, y_te)

shuffled_scores.mean()

## Ringkasan

`cross_val_score` memberi gambaran performa yang lebih stabil daripada satu pembagian data. `KFold` berguna saat alur pengolahan perlu dikendalikan sendiri, dan `Pipeline` menjaga agar standardisasi tetap dihitung dari data latih setiap lipatan.